# Competition Workbench

*one explicit contract, from unfamiliar data to a reproducible submission*

## The question

This is the production-hardening reference after a basic analysis works. It supports IID,
grouped, time-series and panel geometries; regression, binary classification and ranking;
strict OOF evidence; sealed holdout; validated submission; and a reloadable evidence bundle.
For a timed financial exercise, begin with the [volatility project workflow](README.md#volatility-project-workflow).

## Project-to-production route

The shortest useful path here is [R9.1](#r9-1), [R9.3](#r9-3), [R9.5](#r9-5),
[R9.8](#r9-8), [R9.10](#r9-10)–[R9.13](#r9-13), [R9.15](#r9-15) and
[R9.17](#r9-17)–[R9.18](#r9-18): contract, roles, audit, folds, pipeline, baseline,
OOF score, condition diagnostic, locked holdout and checked submission.

Input fingerprints, adversarial validation, null controls, model explanation, atomic bundle
export and the full condition matrix ([R9.4](#r9-4), [R9.7](#r9-7), [R9.14](#r9-14),
[R9.16](#r9-16), [R9.19](#r9-19), [R9.20](#r9-20)) are optional hardening steps once
the core result is stable.

> **Provenance:** stored outputs describe the data and paths printed in that run. Rerun top
> to bottom before treating them as results for the files currently available.

In [1]:
try:
    import rvlab
except ModuleNotFoundError:
    import pathlib, sys
    _root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
                 if (p / "src" / "rvlab").is_dir())
    sys.path.insert(0, str(_root / "src"))
    import rvlab
rvlab.setup_notebook()

roughvol,/Users/xiaohao/rough_pricing_env/Rough-Pricing/src
research_shared,/Users/xiaohao/Effective_Engine/MVP/demo/python/research/shared
● smile_90d,"Multi-maturity: T in [0.019, 0.164]. The term-structure dataset."
● smile_127d,Near-tenor only: T ~ 0.02-0.03. The long-history forecasting dataset.
● smile_5d,Near-tenor only.
● chain_panel,"2025-08-07 to 2025-08-13, one expiry per bar."


## Parameters

In [2]:
PROFILE = "time"             # iid | grouped | time | panel
TASK = "regression"           # regression | binary | ranking (ranking: grouped/panel only)
METRIC = "root_mean_squared_error"
METRIC_DIRECTION = "minimize" # checked against the supported metric registry
SUBMISSION_TRANSFORM = "identity"  # identity | rank_descending_zero
RANKING_TARGET_ORDER = "higher_is_better"  # higher_is_better | lower_is_better
TEST_TIME_RELATION = "future"  # future | unrestricted; required for time/panel
POSITIVE_LABEL = 1
USE_DEMO = True                 # never silently replace missing competition data
TARGET = "target"
ID_COLUMNS = ("row_id",)
FEATURES = ("x_num", "x_aux", "category")
FEATURE_AVAILABILITY = {"x_num": True, "x_aux": True, "category": True}
N_SPLITS = 4
FINAL_HOLDOUT_FRACTION = 0.20
LABEL_HORIZON = 1               # future steps used by one label; drives the purge
FEATURE_LOOKBACK = 2            # precomputed past-only history; does not drive the purge
EXPLICIT_GAP = None             # may exceed, never understate, LABEL_HORIZON
RIDGE_ALPHA = 1.0
LOGISTIC_C = 1.0
DRIFT_SPLITS = 4
NULL_REPEATS, NULL_TOLERANCE = 5, 0.10
ALLOW_CROSS_SPLIT_DUPLICATES = False
N_TRAIN, N_TEST = 240, 80
SEED = 42
OUTPUT_NAME = "competition_workbench"
NOTEBOOK_SOURCE_PATH = None  # optional path to this running .ipynb for code identity
DATA_LOADER_IDENTITY = "rvlab.make_competition_demo:v1"  # immutable loader/dataset version; required

## Where the data comes from

This is the only data-loading cell. It deliberately uses an explicit demo switch: a typo in
a real competition path must fail, not produce a convincing synthetic score. Replace the demo
call with your `train`, `test`, and `sample_submission` loaders, then map their role columns
into `ProblemSpec`. For real data, replace `DATA_LOADER_IDENTITY` with an immutable dataset slug/version plus loader-code version. Downstream cells should not change.

In [3]:
from rvlab.competition import ProblemSpec, make_competition_demo, make_final_holdout

if not USE_DEMO:
    raise RuntimeError("Replace the swap-data cell with real loaders before USE_DEMO=False")
train, test, sample_submission = make_competition_demo(
    PROFILE, task=TASK, n_train=N_TRAIN, n_test=N_TEST, random_state=SEED)
geometry_roles = {
    "iid": {},
    "grouped": {"group_column": "group"},
    "time": {"time_column": "time"},
    "panel": {"time_column": "time", "group_column": "group"},
}
chronological = PROFILE in {"time", "panel"}
spec = ProblemSpec(
    geometry=PROFILE, task=TASK, target=TARGET, id_columns=ID_COLUMNS, n_splits=N_SPLITS,
    random_state=SEED, label_horizon=LABEL_HORIZON if chronological else 0,
    feature_lookback=FEATURE_LOOKBACK if chronological else 0,
    test_time_relation=TEST_TIME_RELATION if chronological else None,
    gap=EXPLICIT_GAP if chronological else None, positive_label=POSITIVE_LABEL,
    metric=METRIC, metric_direction=METRIC_DIRECTION,
    submission_transform=SUBMISSION_TRANSFORM,
    ranking_target_order=RANKING_TARGET_ORDER if TASK == "ranking" else None,
    **geometry_roles[PROFILE])
features = list(FEATURES)
development_positions, final_holdout_positions = make_final_holdout(
    train, spec, fraction=FINAL_HOLDOUT_FRACTION)
development = train.iloc[development_positions].copy()
print(f"{PROFILE}/{spec.task}: train={train.shape}, test={test.shape}, synthetic={train.attrs.get('synthetic', False)}")
train.drop(columns=[spec.target]).head(3)

time/regression: train=(240, 6), test=(80, 5), synthetic=True


,row_id,time,x_num,x_aux,category
0,train_000000,2020-01-01,0.6568,-1.4516,mid
1,train_000001,2020-01-02,-1.1258,NaN,low
2,train_000002,2020-01-03,1.2775,1.5848,high


## Recipes in this notebook

**Contract and data boundary**

| Recipe | What it does |
|---|---|
| [R9.1](#r9-1) | Make the problem contract immutable |
| [R9.2](#r9-2) | Route by data geometry, not model preference |
| [R9.3](#r9-3) | Separate roles from features and state availability |
| [R9.4](#r9-4) | Fingerprint the exact inputs |

**Audit and validation design**

| Recipe | What it does |
|---|---|
| [R9.5](#r9-5) | Run integrity gates before EDA |
| [R9.6](#r9-6) | Compare train and test one feature at a time |
| [R9.7](#r9-7) | Detect multivariate drift adversarially |
| [R9.8](#r9-8) | Audit the sealed development set, then build folds |
| [R9.9](#r9-9) | Prove the folds do not leak |

**Out-of-fold evidence**

| Recipe | What it does |
|---|---|
| [R9.10](#r9-10) | Keep every fitted transform inside a pipeline |
| [R9.11](#r9-11) | Fit the trivial baseline inside each fold |
| [R9.12](#r9-12) | Make one OOF ledger the source of truth |
| [R9.13](#r9-13) | Score the exact objective and its worst fold |
| [R9.14](#r9-14) | Demand that a permuted target fails |
| [R9.15](#r9-15) | Read the diagnostic for this condition |
| [R9.16](#r9-16) | Explain one held-out validation fold safely |

**Refit and submission**

| Recipe | What it does |
|---|---|
| [R9.17](#r9-17) | Lock, open the final holdout once, then refit |
| [R9.18](#r9-18) | Validate against the sample submission |
| [R9.19](#r9-19) | Save a reloadable evidence bundle |
| [R9.20](#r9-20) | Smoke-test every supported geometry/task condition |

## Things to watch

The model is rarely the most dangerous part of a competition notebook.

**Python and pandas**

* **Never infer roles from dtype alone.** Integer IDs are not numeric features; string dates
  are not categories. Name roles once and exclude them explicitly.
* **A pandas index is not a submission contract.** Predictions are positional here because
  the sample submission owns row order; IDs are checked before values are attached.
* **Keep `fit` inside the fold.** Imputation, clipping, scaling, encoding, resampling, feature
  selection and calibration all learn state. If a step learns state, it belongs in a pipeline.

**Analysis**

* **Geometry chooses validation.** Repeated customers need group-exclusive folds; future
  predictions need chronological folds; panels split whole dates, not rows.
* **Past-only lookback is not a purge.** The forward label horizon drives boundary purging.
  Lookback instead determines how much history inference must carry.
* **The public leaderboard is not a validation fold.** Choose features, thresholds, models
  and blends on OOF evidence; touch the competition test data only for inference and drift.
* **Synthetic data is a pipeline test, not evidence.** It proves the machinery runs and can
  recover planted signal. It says nothing about the real task.

## Contract and data boundary

The specification below is the notebook's constitution. Changing it creates a different run.

## R9.1 · Make the problem contract immutable
<a id="r9-1"></a>

**Topic** Experiment design

**Use when** starting any supervised analysis. The contract names the unit of independence, target, IDs, prediction geometry, metric and information horizons before a score can influence those choices.

**Inputs** `spec`, created in the swap-data cell.

**Needs** nothing beyond the setup and data cells

**Gotchas** `ProblemSpec` is frozen on purpose. A changed target, gap, metric or positive label is a new experiment and must produce a new manifest.

**See also** [R9.8](#r9-8) (routing folds) · [R9.19](#r9-19) (manifest)

In [4]:
import dataclasses, pandas as pd

pd.Series(dataclasses.asdict(spec), name="declared value").to_frame()

,declared value
geometry,time
task,regression
target,target
id_columns,"(row_id,)"
time_column,time
group_column,None
test_time_relation,future
n_splits,4
random_state,42
label_horizon,1


## R9.2 · Route by data geometry, not model preference
<a id="r9-2"></a>

**Topic** Validation design

**Use when** deciding how rows may cross a validation boundary. The question is which observations share information, not which splitter is fashionable.

**Inputs** a description of how rows are generated.

**Needs** nothing beyond the setup and data cells

**Gotchas** a panel is not ordinary grouped data: the same entity is expected on both sides, while the same timestamp must never be on both sides.

**See also** [R3.11](forecasting_the_smile.ipynb#r3-11) (time splitters) · [R5.16](the_cross_section.ipynb#r5-16) (date folds)

In [5]:
import pandas as pd

pd.DataFrame([
    ("iid", "exchangeable rows", "KFold / StratifiedKFold", "every row once"),
    ("grouped", "repeated person/site", "GroupKFold / StratifiedGroupKFold", "no shared group"),
    ("time", "predict a later time", "expanding TimeSeriesSplit", "train before valid + purge"),
    ("panel", "many entities per time", "whole-time expanding folds", "no shared time + purge"),
], columns=["geometry", "use when", "router", "invariant"]).set_index("geometry")

,use when,router,invariant
geometry,,,
iid,exchangeable rows,KFold / StratifiedKFold,every row once
grouped,repeated person/site,GroupKFold / StratifiedGroupKFold,no shared group
time,predict a later time,expanding TimeSeriesSplit,train before valid + purge
panel,many entities per time,whole-time expanding folds,no shared time + purge


## R9.3 · Separate roles from features and state availability
<a id="r9-3"></a>

**Topic** Leakage

**Use when** turning raw columns into a design matrix. IDs, time, groups, metadata and the target are roles; only columns available at the prediction moment may be features.

**Inputs** `train`, `spec`, and `features`.

**Needs** nothing beyond the setup and data cells

**Gotchas** this core starts from precomputed causal features; it does not build lags. A plausible name is not evidence of availability. Declare every selected feature explicitly and test the upstream feature builder separately.

**See also** [R3.2](forecasting_the_smile.ipynb#r3-2) (four leaks) · [R6.21](where_the_numbers_come_from.ipynb#r6-21) (leakage scan)

In [6]:
import pandas as pd

roles = {c: "feature" for c in features}
declared_roles = {spec.target: "target", **{c: "id" for c in spec.id_columns}}
declared_roles.update({c: role for c, role in
                       ((spec.time_column, "time"), (spec.group_column, "group")) if c})
overlap = sorted(set(features) & set(declared_roles))
assert not overlap, f"roles cannot also be raw features: {overlap}"
assert set(FEATURE_AVAILABILITY) == set(features), "declare availability for every feature"
assert all(v is True for v in FEATURE_AVAILABILITY.values()), "a selected feature is unavailable"
roles.update(declared_roles)  # roles win; a target can never render as a feature
registry = pd.DataFrame({
    "column": train.columns,
    "role": [roles.get(c, "metadata / exclude") for c in train.columns],
    "dtype": train.dtypes.astype(str).to_numpy(),
    "available_at_prediction": [FEATURE_AVAILABILITY.get(c, pd.NA) for c in train.columns],
})
registry

,column,role,dtype,available_at_prediction
0,row_id,id,object,<NA>
1,time,time,datetime64[ns],<NA>
2,x_num,feature,float64,True
3,x_aux,feature,float64,True
4,category,feature,string,True
5,target,target,float64,<NA>


## R9.4 · Fingerprint the exact inputs
<a id="r9-4"></a>

**Topic** Provenance

**Use when** a result may need to be reproduced after files, row order or dtypes change. The hash covers schema, values, index and order.

**Inputs** `train`, `test`, and `sample_submission`.

**Needs** nothing beyond the setup and data cells

**Gotchas** a filename is not an identity. Competition organizers can replace a file without changing its name, and concatenation can reorder identical rows.

**See also** [R1.7](the_volatility_surface.ipynb#r1-7) (frame provenance) · [R5.36](the_cross_section.ipynb#r5-36) (run manifest)

In [7]:
from rvlab.competition import data_fingerprint

fingerprints = {"train": data_fingerprint(train), "test": data_fingerprint(test),
                "sample_submission": data_fingerprint(sample_submission)}
{name: value[:16] + "…" for name, value in fingerprints.items()}

{'train': 'ca6ff01649f5151a…',
 'test': '2ba73c1cd21cc1cb…',
 'sample_submission': '3c72ce36db3741b8…'}

## Audit and validation design

Audit train, test and the boundary between them. EDA that uses the target belongs on training
folds only; test features may be inspected for schema and drift, never for label-driven choices.

## R9.5 · Run integrity gates before EDA
<a id="r9-5"></a>

**Topic** Health check

**Use when** immediately after loading. Duplicated IDs, infinities, all-null columns and silent schema differences should stop the run before they become modelling decisions.

**Inputs** `train`, `test`, `development`, and `spec`.

**Needs** nothing beyond the setup and data cells

**Gotchas** nulls are not automatically defects; duplicate IDs and non-finite numeric values are. Separate fail-fast contracts from exploratory warnings.

**See also** [R6.14](where_the_numbers_come_from.ipynb#r6-14) (full audit) · [R1.11](the_volatility_surface.ipynb#r1-11) (schema contract)

In [8]:
from rvlab.competition import validate_competition_data
from rvlab.data import health_report

boundary_report = validate_competition_data(
    train, test, sample_submission, features, spec,
    availability=FEATURE_AVAILABILITY,
    allow_cross_split_duplicates=ALLOW_CROSS_SPLIT_DUPLICATES)
display(boundary_report.round(4))
health_report(development, key_cols=spec.id_columns)

,column,dtype,train_null_rate,test_null_rate,null_rate_delta,available_at_prediction
0,x_num,float64,0.0000,0.0000,0.0000,True
1,x_aux,float64,0.0792,0.1750,0.0958,True
2,category,string,0.0250,0.1625,0.1375,True


rows,191.0
columns,6.0
memory_mb,0.03
duplicate_rows,0.0
constant_columns,0.0
columns_with_nulls,2.0
total_null_rate,0.0192


## R9.6 · Compare train and test one feature at a time
<a id="r9-6"></a>

**Topic** Drift

**Use when** test inputs are visible. Compare numeric distributions, missingness and unseen categorical levels; these describe deployment conditions without looking at labels.

**Inputs** `train`, `test`, and `features`.

**Needs** R9.5 — run that first.

**Gotchas** drift is not an instruction to drop a feature. It is a reason to test recent/fold performance and validate any mitigation rather than assume it helps.

**See also** [R6.20](where_the_numbers_come_from.ipynb#r6-20) (KS and PSI) · [R9.7](#r9-7) (multivariate drift)

In [9]:
import pandas as pd
from rvlab.data import drift_report

numeric_features = train[features].select_dtypes("number").columns.tolist()
drift = drift_report(train, test, columns=numeric_features)
categorical_features = [c for c in features if c not in numeric_features]
novelty = []
for col in categorical_features:
    known = set(train[col].dropna().astype(str))
    unseen = ~test[col].dropna().astype(str).isin(known)
    novelty.append({"column": col,
                    "unseen_non_null_rate": unseen.mean() if len(unseen) else 0.0})
display(drift.round(4))
display(boundary_report[["column", "train_null_rate",
                         "test_null_rate", "null_rate_delta"]].round(4))
pd.DataFrame(novelty)

,column,ref_mean,cur_mean,mean_shift_in_sd,sd_ratio,ks_stat,ks_pvalue,psi,drifted
0,x_num,-0.0546,0.8632,0.9440,1.1815,0.3833,0.0000,0.7710,True
1,x_aux,-0.0242,-0.0318,-0.0075,1.1200,0.0810,0.8602,0.0925,False


,column,train_null_rate,test_null_rate,null_rate_delta
0,x_num,0.0000,0.0000,0.0000
1,x_aux,0.0792,0.1750,0.0958
2,category,0.0250,0.1625,0.1375


,column,unseen_non_null_rate
0,category,0.1194


## R9.7 · Detect multivariate drift adversarially
<a id="r9-7"></a>

**Topic** Drift

**Use when** individual features look modestly different but their combination may identify train versus test. Cross-validated domain-classifier AUC turns that question into a number.

**Inputs** `train`, `test`, and `features`.

**Needs** nothing beyond the setup and data cells

**Gotchas** high adversarial AUC diagnoses separability, not model failure. Inspect which features drive it and whether validation reproduces the same deployment shift.

**See also** [R9.6](#r9-6) (univariate drift) · [R4.12](when_a_result_is_real.ipynb#r4-12) (conditional performance)

In [10]:
from rvlab.competition import adversarial_validation_report

adversarial = adversarial_validation_report(
    train, test, features, n_splits=DRIFT_SPLITS, random_state=SEED)
print(f"OOF adversarial AUC = {adversarial.auc:.3f} "
      f"(fold mean {adversarial.mean_fold_auc:.3f} ± {adversarial.std_fold_auc:.3f})")
adversarial.to_frame()

OOF adversarial AUC = 0.838 (fold mean 0.834 ± 0.049)


,fold,auc
0,0,0.8175
1,1,0.8233
2,2,0.7892
3,3,0.9042


## R9.8 · Audit the sealed development set, then build folds
<a id="r9-8"></a>

**Topic** Cross-validation

**Use when** the swap-data cell has already sealed a geometry-safe final holdout before showing any target-bearing output. Run label-aware proxy screening and route model-selection folds only inside the remaining development rows.

**Inputs** the sealed `development`, `development_positions`, `final_holdout_positions`, `train`, `features`, and `spec`.

**Needs** R9.5 — run that first.

**Gotchas** the final holdout positions were fixed in the swap-data cell before its target-free preview and are opened once in R9.17. Any target-aware proxy scan belongs on development labels, never on the full training target. For chronological targets, `label_horizon` purges both the holdout and CV boundaries.

**See also** [R3.12](forecasting_the_smile.ipynb#r3-12) (purge and embargo) · [R5.17](the_cross_section.ipynb#r5-17) (final block)

In [11]:
from rvlab.competition import make_folds, validate_development_target_proxies

target_proxy_report = validate_development_target_proxies(
    development, features, spec)
folds = make_folds(development, spec)
{"development": len(development), "final_holdout": len(final_holdout_positions),
 "purged_boundary": len(train) - len(development) - len(final_holdout_positions),
 "target_proxy_suspects": len(target_proxy_report),
 "cv_folds": [(len(fit), len(valid)) for fit, valid in folds]}

{'development': 191,
 'final_holdout': 48,
 'purged_boundary': 1,
 'target_proxy_suspects': 0,
 'cv_folds': [(38, 38), (76, 38), (114, 38), (152, 38)]}

## R9.9 · Prove the folds do not leak
<a id="r9-9"></a>

**Topic** Validation design

**Use when** before passing `cv=` anywhere. Print sizes, target balance, shared groups/times, chronological spans and the realized gap.

**Inputs** `development`, `spec`, and `folds`.

**Needs** R9.8 — run that first.

**Gotchas** a splitter object is only an intention. Materialized indices and asserted invariants are evidence; save them with the run.

**See also** [R3.11](forecasting_the_smile.ipynb#r3-11) (split audit) · [R5.16](the_cross_section.ipynb#r5-16) (map dates to rows)

In [12]:
import numpy as np
from rvlab.competition import fold_diagnostics

fold_report = fold_diagnostics(development, folds, spec)
valid_positions = np.concatenate([valid for _, valid in folds])
assert len(valid_positions) == len(np.unique(valid_positions))
if spec.geometry in {"iid", "grouped"}:
    assert len(valid_positions) == len(development)
fold_report

,fold,n_train,n_valid,row_overlap,n_train_times,n_valid_times,time_overlap,train_time_end,valid_time_start,gap,train_target_mean,valid_target_mean
0,0,38,38,0,38,38,0,2020-02-07,2020-02-09,1,0.2378,0.4133
1,1,76,38,0,76,38,0,2020-03-16,2020-03-18,1,0.2978,-0.5450
2,2,114,38,0,114,38,0,2020-04-23,2020-04-25,1,0.0283,-0.4258
3,3,152,38,0,152,38,0,2020-05-31,2020-06-02,1,-0.0714,0.2601


## Out-of-fold evidence

No tuning is hidden in this worked spine: one declared linear model faces one fold-local
trivial baseline. Add models only after this ledger and metric behave as expected. If you
tune, nest the search as in [R8.28](measuring_and_trading_volatility.ipynb#r8-28).

## R9.10 · Keep every fitted transform inside a pipeline
<a id="r9-10"></a>

**Topic** Pipelines

**Use when** features mix numbers, categories and missing values. Each cloned fold learns imputation, encoding and scaling from its own training rows only.

**Inputs** `development`, `features`, and `spec.task`.

**Needs** R9.8 — run that first.

**Gotchas** resampling for imbalance also belongs inside the fold. Class weights are used here because naive oversampling before CV duplicates validation information.

**See also** [R3.9](forecasting_the_smile.ipynb#r3-9) (leak-proof pipeline) · [R8.25](measuring_and_trading_volatility.ipynb#r8-25) (target encoding)

In [13]:
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.pipeline import Pipeline
from rvlab.pipelines import make_column_transformer

numeric_features = development[features].select_dtypes("number").columns.tolist()
categorical_features = [c for c in features if c not in numeric_features]
prepare = make_column_transformer(numeric_features, categorical_features)
estimator = (LogisticRegression(C=LOGISTIC_C, class_weight="balanced",
                                max_iter=2000, random_state=SEED)
             if spec.task == "binary" else Ridge(alpha=RIDGE_ALPHA))
pipe = Pipeline([("prepare", prepare), ("model", estimator)])
pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('prepare', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contain

## R9.11 · Fit the trivial baseline inside each fold
<a id="r9-11"></a>

**Topic** Baselines

**Use when** before fitting a learned model. Regression predicts the fold-training mean, binary classification the fold-training prevalence, and ranking a no-information constant.

**Inputs** `development`, `spec`, and `folds`.

**Needs** R9.8 — run that first.

**Gotchas** computing the global mean or prevalence lets validation targets influence their own benchmark. A baseline must obey the same information boundary as the model.

**See also** [R3.10](forecasting_the_smile.ipynb#r3-10) (domain baselines) · [R8.22](measuring_and_trading_volatility.ipynb#r8-22) (class prevalence)

In [14]:
from rvlab.competition import baseline_ledger

baseline = baseline_ledger(development, spec, folds=folds)
baseline.groupby("fold").agg(n=("prediction", "size"),
                                    prediction=("prediction", "first"),
                                    target_mean=("y_true", "mean"))

,n,prediction,target_mean
fold,,,
0,38,0.2378,0.4133
1,38,0.2978,-0.5450
2,38,0.0283,-0.4258
3,38,-0.0714,0.2601


## R9.12 · Make one OOF ledger the source of truth
<a id="r9-12"></a>

**Topic** Model comparison

**Use when** fitting any candidate. The ledger records source row, fold, IDs, time/group roles, truth and prediction so every later diagnostic uses identical rows.

**Inputs** `pipe`, `development`, `features`, `spec`, and `folds`.

**Needs** R9.8, R9.10 — run those first.

**Gotchas** pooled OOF and mean fold scores answer different questions for non-decomposable metrics such as AUC. Use the fold mean for model selection; keep pooled OOF explicitly labelled as a diagnostic.

**See also** [R3.15](forecasting_the_smile.ipynb#r3-15) (model contest) · [R5.27](the_cross_section.ipynb#r5-27) (OOF stacking)

In [15]:
from rvlab.competition import oof_ledger

oof = oof_ledger(pipe, development, features, spec, folds=folds)
assert oof["row_position"].is_unique
print(f"{len(oof):,} validation predictions across {oof['fold'].nunique()} folds")
oof.head(3)

152 validation predictions across 4 folds


,row_position,fold,row_id,time,y_true,prediction
0,39,0,train_000039,2020-02-09,3.3325,0.4956
1,40,0,train_000040,2020-02-10,0.9420,1.1578
2,41,0,train_000041,2020-02-11,1.9642,0.3583


## R9.13 · Score the exact objective and its worst fold
<a id="r9-13"></a>

**Topic** Metrics

**Use when** comparing candidates. The checked metric and direction in `ProblemSpec` are applied fold by fold; mean, dispersion and worst fold are primary, with pooled OOF reported separately.

**Inputs** `oof`, `baseline`, and `spec`.

**Needs** R9.11, R9.12 — run those first.

**Gotchas** unsupported metric names fail at contract construction. Extend the tested metric registry before using a custom implementation; never relabel a hard-coded score.

**See also** [R5.18](the_cross_section.ipynb#r5-18) (ranking objective) · [R3.20](forecasting_the_smile.ipynb#r3-20) (classification metrics)

In [16]:
import pandas as pd
from rvlab.competition import score_ledger

score_reports = {"baseline": score_ledger(baseline, spec),
                 "model": score_ledger(oof, spec)}
score_table = pd.concat([report.to_frame().assign(candidate=name)
                         for name, report in score_reports.items()], ignore_index=True)
score_table = score_table.set_index("candidate")
fold_score_table = pd.concat([report.folds_frame().assign(candidate=name)
                              for name, report in score_reports.items()], ignore_index=True)
score_table

,metric,direction,cv_mean,cv_std,worst_fold,pooled_oof
candidate,,,,,,
baseline,root_mean_squared_error,minimize,1.9525,0.3937,2.4435,1.9820
model,root_mean_squared_error,minimize,0.8598,0.0556,0.9244,0.8612


## R9.14 · Demand that a permuted target fails
<a id="r9-14"></a>

**Topic** Leakage

**Use when** the pipeline first works and after major feature changes. With labels permuted, performance must collapse toward no-information.

**Inputs** `development`, `pipe`, `features`, `spec`, and `folds`.

**Needs** R9.10, R9.13 — run those first.

**Gotchas** IID labels permute globally; ranking labels permute within query. Non-ranking grouped, time, and panel controls use an exact constrained cross-structure permutation, preserving every target value while preventing self-block assignment. If one block exceeds half the rows, that exact null is impossible and fails explicitly.

**See also** [R8.25](measuring_and_trading_volatility.ipynb#r8-25) (random-target leak) · [R2.12](what_roughness_claims.ipynb#r2-12) (positive control)

In [17]:
import pandas as pd
from rvlab.competition import null_control_report

null_report = null_control_report(
    pipe, development, features, spec, folds=folds, n_repeats=NULL_REPEATS,
    random_state=SEED + 1, tolerance=NULL_TOLERANCE, assert_baseline=True)
pd.Series({"model_selection_baseline_cv": score_reports["baseline"].cv_mean,
           "mean_null_fold_local_baseline": null_report["baseline_cv_mean"].mean(),
           "mean_null_cv": null_report["null_cv_mean"].mean(),
           "mean_null_improvement": null_report.attrs["mean_improvement_over_baseline"],
           "allowed_improvement": NULL_TOLERANCE}, name=spec.metric).to_frame()

,root_mean_squared_error
model_selection_baseline_cv,1.9525
mean_null_fold_local_baseline,1.8437
mean_null_cv,1.9162
mean_null_improvement,-0.0725
allowed_improvement,0.1000


## R9.15 · Read the diagnostic for this condition
<a id="r9-15"></a>

**Topic** Diagnostics

**Use when** CV evidence is known. Compose task diagnostics with geometry diagnostics: a grouped binary problem needs both imbalance/calibration evidence and group stability.

**Inputs** `oof`, `spec`, and `fold_report`.

**Needs** R9.9, R9.13 — run those first.

**Gotchas** subgroup analysis creates multiplicity. Predeclare important slices, report counts, and treat post-hoc discoveries as hypotheses for the next holdout.

**See also** [R8.22](measuring_and_trading_volatility.ipynb#r8-22) (imbalance) · [R3.22](forecasting_the_smile.ipynb#r3-22) (score path)

In [18]:
import pandas as pd
from sklearn.metrics import average_precision_score
from rvlab.evaluate import calibration_report, imbalance_report
diagnostics = {"fold_path": score_reports["model"].folds_frame()}
if spec.task == "binary":
    y_binary = (oof["y_true"] == spec.positive_label).astype(int)
    task_report = pd.concat([imbalance_report(y_binary), calibration_report(y_binary, oof["prediction"])])
    task_report["average_precision"] = average_precision_score(y_binary, oof["prediction"])
    diagnostics["binary"] = task_report.to_frame("value")
elif spec.task == "ranking":
    query = spec.time_column or spec.group_column
    diagnostics["rank_by_query"] = (oof.groupby(query, observed=True)[["y_true", "prediction"]]
        .apply(lambda g: g["y_true"].corr(g["prediction"], method="spearman"))
        .describe().to_frame("rank_ic"))
else:
    diagnostics["residual"] = (oof["y_true"] - oof["prediction"]).describe().to_frame()
if spec.group_column:
    slice_truth = y_binary if spec.task == "binary" else pd.to_numeric(oof["y_true"])
    slices = oof.assign(_truth=slice_truth, abs_error=(slice_truth - oof["prediction"]).abs())
    diagnostics["group_slices"] = slices.groupby(spec.group_column, observed=True).agg(
        n=("prediction", "size"), target_mean=("_truth", "mean"),
        prediction_mean=("prediction", "mean"), mae=("abs_error", "mean")).describe()
for name, frame in diagnostics.items():
    print(name); display(frame)
list(diagnostics)

fold_path


,fold,score
0,0,0.8135
1,1,0.8134
2,2,0.8879
3,3,0.9244


residual


,0
count,152.0000
mean,-0.0968
std,0.8585
min,-2.5253
25%,-0.7357
50%,-0.0925
75%,0.4808
max,2.8368


['fold_path', 'residual']

## R9.16 · Explain one held-out validation fold safely
<a id="r9-16"></a>

**Topic** Interpretability

**Use when** asking which raw columns the fitted pipeline relies on. Permute columns in one held-out CV fold and measure degradation with the declared metric.

**Inputs** `pipe`, `development`, `features`, `spec`, and `folds`.

**Needs** R9.8, R9.10 — run those first.

**Gotchas** the fold is held out from its fitted model, not untouched by the analyst. Ranking permutations stay within query; binary scoring uses the declared positive label. Correlated features still share credit, and importance is not causality.

**See also** [R4.1](when_a_result_is_real.ipynb#r4-1) (permutation importance) · [R4.3](when_a_result_is_real.ipynb#r4-3) (correlated groups)

In [19]:
from rvlab.competition import heldout_permutation_importance

heldout_importance = heldout_permutation_importance(
    pipe, development, features, spec, folds=folds, fold=-1,
    n_repeats=8, random_state=SEED)
heldout_importance

,feature,importance,std,base_score,fold,n_valid
0,x_num,1.0318,0.2016,0.9244,3,38
1,x_aux,0.2759,0.1032,0.9244,3,38
2,category,0.1041,0.0250,0.9244,3,38


## Refit and submission

At this boundary the experiment stops changing. Record the selected configuration first;
then train on all eligible labels exactly once and treat test rows as inference only.

## R9.17 · Lock, open the final holdout once, then refit
<a id="r9-17"></a>

**Topic** Inference

**Use when** model selection is finished. Score the reserved geometry-safe holdout once, then refit the unchanged pipeline on all labels.

**Inputs** `pipe`, `development`, `final_holdout_positions`, `train`, `test`, `features`, and `spec`.

**Needs** R9.10, R9.13 — run those first.

**Gotchas** no choice may change after seeing the final holdout. This core requires precomputed causal train/test features with identical names, order and dtypes; history-buffer parity belongs in the upstream feature builder.

**See also** [R5.31](the_cross_section.ipynb#r5-31) (refit once) · [R5.32](the_cross_section.ipynb#r5-32) (schema alignment)

In [20]:
import pandas as pd
from sklearn.base import clone
from rvlab.competition import score_predictions

selection_lock = {"model": type(estimator).__name__, "metric": spec.metric,
                  "direction": spec.metric_direction,
                  "submission_transform": spec.submission_transform,
                  "ridge_alpha": RIDGE_ALPHA, "logistic_c": LOGISTIC_C,
                  "features": features.copy()}
def predict_for_task(model, frame):
    if spec.task != "binary":
        return model.predict(frame)
    classes = list(model.classes_)
    return model.predict_proba(frame)[:, classes.index(spec.positive_label)]
holdout = train.iloc[final_holdout_positions]
holdout_model = clone(pipe).fit(development[features], development[spec.target])
trace_columns = list(dict.fromkeys([*spec.id_columns, *[c for c in (spec.time_column, spec.group_column) if c]]))
holdout_evidence = holdout[trace_columns].reset_index(drop=True)
holdout_evidence["y_true"] = holdout[spec.target].to_numpy()
holdout_evidence["prediction"] = predict_for_task(holdout_model, holdout[features])
final_holdout_score = score_predictions(holdout_evidence, spec)
assert train[features].dtypes.astype(str).equals(test[features].dtypes.astype(str))
test_X = test.loc[:, features].copy()  # strict: never synthesize a missing feature
final_model = clone(pipe).fit(train[features], train[spec.target])
{**selection_lock, "final_holdout_score": final_holdout_score}

{'model': 'Ridge',
 'metric': 'root_mean_squared_error',
 'direction': 'minimize',
 'submission_transform': 'identity',
 'ridge_alpha': 1.0,
 'logistic_c': 1.0,
 'features': ['x_num', 'x_aux', 'category'],
 'final_holdout_score': 0.7861871826181037}

## R9.18 · Validate against the sample submission
<a id="r9-18"></a>

**Topic** Inference

**Use when** producing the grader-facing file. The sample owns column/ID order; the declared task-aware transform converts raw scores to probabilities or within-query ranks before attachment.

**Inputs** `final_model`, `test_X`, `test`, `sample_submission`, and `spec`.

**Needs** R9.17 — run that first.

**Gotchas** ranking conventions differ. Declare `identity` or zero-based descending within-query ranks in the immutable contract, including tie behaviour; never improvise after seeing scores.

**See also** [R5.34](the_cross_section.ipynb#r5-34) (ranking submission) · [R5.35](the_cross_section.ipynb#r5-35) (streaming parity)

In [21]:
import numpy as np
from rvlab.competition import validate_submission

test_prediction = predict_for_task(final_model, test_X)
submission = validate_submission(test, test_prediction, sample_submission, spec)
assert np.isfinite(submission[spec.prediction_column]).all()
print(f"submission: {submission.shape}, IDs unique, order preserved, values finite")
submission.head(3)

submission: (80, 2), IDs unique, order preserved, values finite


,row_id,target
0,test_000000,-0.7899
1,test_000001,0.5974
2,test_000002,-1.2746


## R9.19 · Save a reloadable evidence bundle
<a id="r9-19"></a>

**Topic** Reproducibility

**Use when** a model or submission leaves the notebook. Save folds, OOF predictions, metrics, feature order, selected configuration, input hashes, environment, fitted pipeline and model card together.

**Inputs** all locked run artifacts from R9.9–R9.18.

**Needs** R9.5, R9.6, R9.7, R9.8, R9.14, R9.15, R9.16, R9.18 — run those first.

**Gotchas** serialization alone proves little. Stage, fresh-process reload, and validate every file before an atomic publish; the final identity includes every artifact hash. Configure `NOTEBOOK_SOURCE_PATH` when discoverable and always replace the demo loader identity for real data; the manifest records whether notebook source was fingerprinted.

**See also** [R4.21](when_a_result_is_real.ipynb#r4-21) (export together) · [R5.36](the_cross_section.ipynb#r5-36) (manifest)

In [22]:
import pandas as pd
oof_export = pd.concat(
    [baseline.assign(candidate="baseline"), oof.assign(candidate="model")],
    ignore_index=True)
oof_export = oof_export[["candidate", *baseline.columns]]
tables = {
    "oof_predictions.csv": oof_export, "cv_scores.csv": score_table.reset_index(),
    "fold_scores.csv": fold_score_table, "null_controls.csv": null_report,
    "development_target_proxy_scan.csv": target_proxy_report,
    "missingness_schema.csv": boundary_report, "numeric_drift.csv": drift,
    "categorical_novelty.csv": pd.DataFrame(novelty),
    "adversarial_drift.csv": adversarial.to_frame().assign(overall_auc=adversarial.auc),
    "heldout_importance.csv": heldout_importance,
    "final_holdout_score.csv": pd.DataFrame([{
        "metric": spec.metric, "direction": spec.metric_direction,
        "score": final_holdout_score}]),
    "final_holdout_predictions.csv": holdout_evidence.assign(source_position=final_holdout_positions),
}
tables.update({f"diagnostic_{name}.csv": frame for name, frame in diagnostics.items()})
sorted(tables)

['adversarial_drift.csv',
 'categorical_novelty.csv',
 'cv_scores.csv',
 'development_target_proxy_scan.csv',
 'diagnostic_fold_path.csv',
 'diagnostic_residual.csv',
 'final_holdout_predictions.csv',
 'final_holdout_score.csv',
 'fold_scores.csv',
 'heldout_importance.csv',
 'missingness_schema.csv',
 'null_controls.csv',
 'numeric_drift.csv',
 'oof_predictions.csv']

The evidence inventory is fixed above. Next freeze the complete run-parameter and loader contract; this remains available even when a hosted environment cannot expose the running notebook file.

In [23]:
import dataclasses, rvlab.competition as competition_module
if not isinstance(DATA_LOADER_IDENTITY, str) or not DATA_LOADER_IDENTITY.strip():
    raise ValueError("DATA_LOADER_IDENTITY must name an immutable loader/dataset version")
if not train.attrs.get("synthetic", False) and DATA_LOADER_IDENTITY.strip() == "rvlab.make_competition_demo:v1":
    raise ValueError("replace the demo DATA_LOADER_IDENTITY for real competition data")
run_parameters = {
    "spec": dataclasses.asdict(spec), "feature_availability": dict(FEATURE_AVAILABILITY),
    "final_holdout_fraction": FINAL_HOLDOUT_FRACTION, "model": {"ridge_alpha": RIDGE_ALPHA, "logistic_c": LOGISTIC_C},
    "drift_splits": DRIFT_SPLITS, "null": {"repeats": NULL_REPEATS, "tolerance": NULL_TOLERANCE},
    "allow_cross_split_duplicates": ALLOW_CROSS_SPLIT_DUPLICATES, "demo": {"enabled": USE_DEMO, "n_train": N_TRAIN, "n_test": N_TEST}}
notebook_source = competition_module.resolve_notebook_source("competition_workbench.ipynb", NOTEBOOK_SOURCE_PATH)
model_card = {
    "estimator": final_model.named_steps["model"].__class__.__name__, "features": features, "selection": selection_lock,
    "cv": score_table.reset_index().to_dict("records"), "final_holdout_score": float(final_holdout_score),
    "null_mean_improvement": float(null_report.attrs["mean_improvement_over_baseline"]), "adversarial_auc": float(adversarial.auc),
    "training_data": "synthetic demo" if train.attrs.get("synthetic", False) else "declared real data", "scope": "precomputed causal structured-data features",
    "data_loader_identity": DATA_LOADER_IDENTITY.strip(), "notebook_source_fingerprinted": notebook_source is not None,
    "run_parameters": run_parameters,
    "limitations": ["OOF is selection evidence", "final holdout opened once", "test drift inspection is transductive"]}
model_card

{'estimator': 'Ridge',
 'features': ['x_num', 'x_aux', 'category'],
 'selection': {'model': 'Ridge',
  'metric': 'root_mean_squared_error',
  'direction': 'minimize',
  'submission_transform': 'identity',
  'ridge_alpha': 1.0,
  'logistic_c': 1.0,
  'features': ['x_num', 'x_aux', 'category']},
 'cv': [{'candidate': 'baseline',
   'metric': 'root_mean_squared_error',
   'direction': 'minimize',
   'cv_mean': 1.9524901331328839,
   'cv_std': 0.3936942852653598,
   'worst_fold': 2.443478951192514,
   'pooled_oof': 1.9820353459686884},
  {'candidate': 'model',
   'metric': 'root_mean_squared_error',
   'direction': 'minimize',
   'cv_mean': 0.859805992942901,
   'cv_std': 0.0555717014675698,
   'worst_fold': 0.9244118685134198,
   'pooled_oof': 0.8611518484004598}],
 'final_holdout_score': 0.7861871826181037,
 'null_mean_improvement': -0.07250438332040758,
 'adversarial_auc': 0.8375000000000001,
 'training_data': 'synthetic demo',
 'scope': 'precomputed causal structured-data features',
 '

Finally fingerprint the training-pipeline and data-quality implementations (plus the notebook when available), then stage, fresh-process reload, validate, hash, and atomically publish the bundle.

In [24]:
import inspect
import rvlab.data.quality as quality_module
import rvlab.evaluate.classification as classification_module
import rvlab.pipelines.build as pipeline_module
from rvlab.config import OUTPUT_DIR, REPO_ROOT
code_paths = [inspect.getfile(module) for module in (
    competition_module, pipeline_module, quality_module, classification_module)]
code_paths += [notebook_source] if notebook_source else []
run_dir = competition_module.export_run_bundle(
    OUTPUT_DIR, OUTPUT_NAME, spec, train, test, sample_submission,
    features, final_model, submission, development=development, folds=folds,
    source_positions=development_positions, tables=tables,
    model_card=model_card, code_paths=code_paths,
    repo_root=REPO_ROOT if REPO_ROOT.is_dir() else None)
sorted(path.name for path in run_dir.iterdir())

['adversarial_drift.csv',
 'categorical_novelty.csv',
 'cv_scores.csv',
 'development_target_proxy_scan.csv',
 'diagnostic_fold_path.csv',
 'diagnostic_residual.csv',
 'final_holdout_predictions.csv',
 'final_holdout_score.csv',
 'fold_assignments.csv',
 'fold_scores.csv',
 'heldout_importance.csv',
 'manifest.json',
 'missingness_schema.csv',
 'model.joblib',
 'model_card.json',
 'null_controls.csv',
 'numeric_drift.csv',
 'oof_predictions.csv',
 'split_diagnostics.csv',
 'submission.csv',
 'table_metadata.json']

## R9.20 · Smoke-test every supported geometry/task condition
<a id="r9-20"></a>

**Topic** Validation design

**Use when** changing the router or adapting the template. The matrix exercises boundary gates, final-holdout routing, folds, OOF scoring, refit, task-aware postprocessing and submission validation for every supported geometry/task pairing.

**Inputs** no competition data; deterministic demo profiles are generated locally.

**Needs** nothing beyond the setup and data cells

**Gotchas** this is a positive control for plumbing, not evidence about a model. Keep the real-data run identity separate from every synthetic smoke result.

**See also** [R1.3](the_volatility_surface.ipynb#r1-3) (synthetic mode) · [R2.12](what_roughness_claims.ipynb#r2-12) (known-truth control)

In [25]:
from rvlab.competition import smoke_test_condition_matrix
condition_matrix = smoke_test_condition_matrix(random_state=SEED)
condition_matrix.set_index("condition")

,geometry,task,metric,direction,folds,max_row_overlap,oof_rows,baseline_cv_mean,cv_mean,cv_std,worst_fold,pooled_oof,final_holdout_score,submission_rows,postprocess
condition,,,,,,,,,,,,,,,
iid/regression,iid,regression,root_mean_squared_error,minimize,3,0,144,1.8964,0.7422,0.1530,0.8972,0.7526,0.9046,60,identity
iid/binary,iid,binary,roc_auc,maximize,3,0,144,0.5000,0.7784,0.0841,0.6953,0.7821,0.8571,60,identity
grouped/regression,grouped,regression,root_mean_squared_error,minimize,3,0,138,2.1934,0.8326,0.1988,1.0599,0.8571,0.8618,60,identity
grouped/binary,grouped,binary,roc_auc,maximize,3,0,138,0.5000,0.8475,0.0116,0.8384,0.8458,0.9151,60,identity
grouped/ranking,grouped,ranking,spearman,maximize,3,0,138,0.0000,0.9396,0.0444,0.8929,0.9423,0.9724,60,rank_descending_zero
time/regression,time,regression,root_mean_squared_error,minimize,3,0,105,1.9603,0.8416,0.1031,0.9568,0.8458,0.8158,60,identity
time/binary,time,binary,roc_auc,maximize,3,0,105,0.5000,0.7064,0.0969,0.6034,0.6852,0.9264,60,identity
panel/regression,panel,regression,root_mean_squared_error,minimize,3,0,90,1.7300,0.8918,0.1101,1.0107,0.8963,0.9776,60,identity
panel/binary,panel,binary,roc_auc,maximize,3,0,90,0.5000,0.8551,0.0614,0.7900,0.8529,0.8833,60,identity


## Verdict & what carries forward

The workbench turns the series into one reusable, testable spine:

`contract → load explicitly → assign roles → fingerprint → audit → measure drift → reserve a
geometry-safe final holdout → route and assert development folds → fit a fold-local baseline →
collect one OOF ledger → score fold mean/std/worst plus pooled diagnostic → run repeated null
controls → inspect condition-specific failure → lock → open the final holdout once → refit on
all labels → task-aware postprocess → validate the sample → reload for parity → save evidence`

What changes across competitions is now visible and localized: the swap-data adapter, the
problem specification, the feature builder, the estimator, and the exact metric. What must not
change is the information boundary.

Before making a performance claim, answer yes to six questions: was a final holdout reserved
before model selection; did the splitter match the geometry; did every learned transform fit
inside its fold; did the model beat a fold-local baseline on the exact metric; did repeated
negative controls collapse; and did the locked pipeline survive its one-time final holdout? A
complete release must also reproduce a sample-valid submission in original row order after
reload. If any answer is no, the run is an experiment, not a result.